## The performance of a lock complex
In this notebook, we simulate the same lock complex, but now we will generate vessels from a fleet composition with an exponential interarrival time distribution.

To run the simulation smoothly and repeatedly, we create functions for each simulation construction step. 
We start by importing the required packages:

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim

# import of basic core mixins and utils for creating objects, inspecting the output and plotting
from opentnsim.core import Movable, Identifiable, Locatable, Routable, Log, VesselProperties
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.utils import create_object, generate_vessels_from_distribution
from opentnsim.core.visualizations import generate_vessel_gantt_chart

# import of utils for graph visualization
from opentnsim.graph.visualizations import plot_graph

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable
from opentnsim.lock.calculations import estimate_lock_capacity, calculate_lock_occupancy
from opentnsim.lock.logutils import calculate_cycle_information, get_vessel_delays

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
    
print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 2.3.1.dev0+g545dd1b5e.d20260507


#### 0. Create environment

In [2]:
def create_environment(simulation_start):
    # start simpy environment
    env = simpy.Environment(initial_time=simulation_start.timestamp())
    env.epoch = simulation_start
    return env

#### 1. Create graph

In [3]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

def create_graph(env):
    # create a directed graph
    graph = nx.Graph()
    
    # add nodes
    graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
    graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
    
    # add edge
    graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(5000, 0)])), weight=1, length_m=10000)
    
    # add graph to environment
    env.graph = graph

#### 1+ Adding infrastructure

In [4]:
def create_lock_complex(env, lock_length):
    lock_chamber = IsLockChamber(env=env,
                                 lock_length = 400, # in meters
                                 lock_width = 50, # in meters
                                 lock_depth = 10, # in meters
                                 name='Lock',      
                                 edge = ('0','1'),
                                 levelling_time = 600, # in seconds
                                 gate_opening_time = 60, # in seconds
                                 gate_closing_time = 60, # in seconds
                                 distance_from_start_node_to_lock_gate_A = 4800., # lock chamber will be located on the edge center (4.8 - 5.2 km)
                                 distance_from_end_node_to_lock_gate_B = 4800., # lock chamber will be located on the edge center (4.8 - 5.2 km)
                                 sailing_distance_to_crossing_point = 500.) # in meters
    
    # The minimum required input for a lock complex are waiting areas at both sides of the lock
    waiting_area_A = IsLockWaitingArea(env=env,
                                       name = 'Waiting area A',
                                       edge = ('0','1'),
                                       orientation = 0, # orientation indicates that waiting area aligns with start node of lock edge (0) or not (1)
                                       distance_from_edge_start = 4300.) # waiting area A will be located at 500 m from the lock gate
    
    waiting_area_B = IsLockWaitingArea(env=env,
                                       name = 'Waiting area B',
                                       edge = ('1','0'),
                                       orientation = 1, # orientation indicates that waiting area aligns with start node of lock edge (0) or not (1)
                                       distance_from_edge_start = 4300.) # waiting area B will be located at 500 m from the lock gate
    
    lock_complex = IsLockComplex(lock_chambers = [lock_chamber],
                                 waiting_areas = [waiting_area_A, waiting_area_B],
                                 registration_nodes = ['0','1'], # registration nodes is where the vessel registers to the lock planner
                                 env=env, # add the lock complex to the environment
                                 name = 'Lock complex',)
    return lock_complex, lock_chamber, waiting_area_A, waiting_area_B

#### 2. Create agents

In [5]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel", # name of the object
    (
        Identifiable,            # assigns the vessel a unique ID to be traceable
        Locatable,               # allows the vessel to have a location
        Routable,                # allows the vessel to have a route over the network
        Log,                     # allows the vessel to have a logbook to keep track of its actions at what time, which distance, and where (location)
        VesselProperties,        # allows the vessel to have properties, like vessel dimensions
        Movable,                 # allows the object to move, with a fixed speed, while logging this activity
        LockComplexTraversable,  # allows to interact with a lock complex           
    ), 
)

In [6]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

**NEW**: To better illustrate the lock's performance, we add two vessel generators at the network boundaries. From each direction, 100 vessels are generated randomly from an exponential distribution with a user-defined mean interarrival rate. The vessels are sampled from a fleet. We use different seeds to get different arrival times.

In [7]:
fleet_data = pd.DataFrame({'Class': ['small vessel','medium vessel', 'large vessel'], 'Length': [150, 225, 300], 'Beam': [15, 22.5, 30], 'Draught': [4, 6, 8]})
fleet_data

,Class,Length,Beam,Draught
0,small vessel,150,15.0,4
1,medium vessel,225,22.5,6
2,large vessel,300,30.0,8


In [8]:
def generate_vessels(env, mean_interarrival_time, Vessel, fleet_composition, fleet_data):
    upstream_vessels = generate_vessels_from_distribution(env=env,
                                                          VesselClass = Vessel,
                                                          vessel_parameters = {'v':4, 
                                                                               'L':'Length', 
                                                                               'B':'Beam', 
                                                                               'T':'Draught', 
                                                                               'type':'Class'},
                                                          mean_arrival_rate=mean_interarrival_time,
                                                          number_of_vessels=100,
                                                          start_node = '0',
                                                          end_node = '1',
                                                          seed = 123,
                                                          use_fleet = True,
                                                          fleet_composition = fleet_composition,
                                                          fleet_data = fleet_data)
    
    downstream_vessels = generate_vessels_from_distribution(env=env,
                                                            VesselClass = Vessel,
                                                            vessel_parameters = {'v':4, 
                                                                                 'L':'Length', 
                                                                                 'B':'Beam', 
                                                                                 'T':'Draught', 
                                                                                 'type':'Class'},
                                                            mean_arrival_rate=mean_interarrival_time,
                                                            number_of_vessels=100,
                                                            start_node = '1',
                                                            end_node = '0',
                                                            seed = 456,
                                                            use_fleet = True,
                                                            fleet_composition = fleet_composition,
                                                            fleet_data = fleet_data)
    
    vessels = upstream_vessels + downstream_vessels

    for vessel in vessels:
        env.process(mission(env, vessel))

#### 3. Run simulation

In [9]:
def create_and_run_simulation(simulation_start, simulation_stop, lock_length, mean_interarrival_time):
    #0. create environment
    env = create_environment(simulation_start)
    
    #1. create graph
    create_graph(env)
    
    #1+ create lock complex
    lock_complex, lock_chamber, waiting_area_A, waiting_area_B = create_lock_complex(
        env, 
        lock_length = lock_length
    )
    
    #2. generating vessels
    generate_vessels(
        env, 
        mean_interarrival_time = mean_interarrival_time, 
        Vessel = Vessel, 
        fleet_composition = fleet_composition, 
        fleet_data = fleet_data
    )
    
    #3. Running environment
    env.run(until=simulation_stop.timestamp())

    return env, lock_complex, lock_chamber, waiting_area_A, waiting_area_B

#### 4. Estimating the lock performance
We have to perform the following steps:

**STEP 1**: Change model input 

In [ ]:
# Environment input (recommended: run model for one week)
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
simulation_stop = datetime.datetime(2025, 1, 7, 0, 0, 0)

# Infrastructure input
lock_length = 250.

# Vessel input
mean_interarrival_time = 60.
    
fleet_composition = {
    "small vessel":33.3, #%
    "medium vessel":33.3, #%
    "large vessel":33.3, #%
}

**STEP 2**: Determine the **lock capacity** by running the model with a mean interarrival time of 0 seconds, meaning all vessels arrive at t=0.

In [11]:
env, lock_complex, lock_chamber, waiting_area_A, waiting_area_B = create_and_run_simulation(
    simulation_start = simulation_start, 
    simulation_stop = simulation_stop, 
    lock_length = lock_length, 
    mean_interarrival_time = 0.,
)

In [12]:
lock_capacity = lock_chamber.get_aggregated_cycle_information()['Cycle-averaged traffic intensity (I_s_avg)']
print(f'The lock capacity is: {lock_capacity} vessels/hours')

The lock capacity is: 3.56 vessels/hours


**STEP 3**: Determine the **lock intensity**, **IC-ratio**, and **vessel delay** by running the model with the user-defined mean interarrival time

In [13]:
env, lock_complex, lock_chamber, waiting_area_A, waiting_area_B = create_and_run_simulation(
    simulation_start = simulation_start, 
    simulation_stop = simulation_stop, 
    lock_length = lock_length, 
    mean_interarrival_time = mean_interarrival_time,
)

Performance indicators:

In [14]:
lock_intensity = lock_chamber.get_aggregated_cycle_information()['Cycle-averaged traffic intensity (I_s_avg)']
_, _, vessel_delays_causes = get_vessel_delays(lock_chamber)
print(f'The lock capacity is: {lock_intensity} vessels/hours')
print(f'The I/C-ratio is then: {np.round(lock_intensity/lock_capacity,2)}')
print(f'The average vessel delay is: {vessel_delays_causes['average_vessel_delay']}')

The lock capacity is: 2.49 vessels/hours
The I/C-ratio is then: 0.7


In [ ]:
lock_intensities = []
lock_capacities = []
vessel_delays = []
mean_interarrival_time = 60.
lock_lengths = np.arange(150,600,50)
for lock_length in lock_lengths:
    env, lock_complex, lock_chamber, waiting_area_A, waiting_area_B = create_and_run_simulation(
        simulation_start = simulation_start, 
        simulation_stop = simulation_stop, 
        lock_length = lock_length, 
        mean_interarrival_time = 0.,
    )
    lock_capacity = lock_chamber.get_aggregated_cycle_information()['Cycle-averaged traffic intensity (I_s_avg)']
    
    env, lock_complex, lock_chamber, waiting_area_A, waiting_area_B = create_and_run_simulation(
        simulation_start = simulation_start, 
        simulation_stop = simulation_stop, 
        lock_length = lock_length, 
        mean_interarrival_time = mean_interarrival_time,
    )
    
    lock_intensity = lock_chamber.get_aggregated_cycle_information()['Cycle-averaged traffic intensity (I_s_avg)']
    lock_intensities.append(lock_intensity)
    lock_capacities.append(lock_capacity)

    _, _, vessel_delays_causes = get_vessel_delays(lock_chamber)
    vessel_delays.append(vessel_delays_causes['average_vessel_delay'].total_seconds()/60)

In [ ]:
fig, ax = plt.subplots()
ax.plot(lock_lengths, np.array(lock_intensities)/np.array(lock_capacities),color='C0',marker='o',label='I/C')
ax.set_xlabel('Lock length [m]')
ax.set_ylabel('I/C-ratio [-]')
ax = ax.twinx()
ax.plot(lock_lengths,vessel_delays,color='C1',marker='o',label='Delay')
ax.set_ylabel('Average vessel delay [min]')
fig.legend();